In [1]:
# ===========================================
# 0) Setup (Colab) — installs & imports
# ===========================================
!pip -q install osmnx==1.9.3 geopandas shapely pyproj fiona

import os
import csv
import math
import gzip
import random
import numpy as np
import pandas as pd
import networkx as nx
import osmnx as ox
from datetime import datetime, timedelta
from tqdm.auto import tqdm

ox.settings.use_cache = True
ox.settings.log_console = False  # set True for more logs

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
d:\OneDrive\Desktop\AUST\Thesis_docs\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# ===========================================
# 1) Parameters — Dhaka-optimized
# ===========================================
PLACE = "Dhaka, Bangladesh"
NETWORK_TYPE = "drive"

# ---- Simulate SEVEN DAYS (15-min bins) ----
SIM_START_DATE = "2025-02-15"     # start date of the 7-day window
FREQ = "15min"
TZ = "Asia/Dhaka"
STEPS_PER_DAY = int(pd.Timedelta("1D") / pd.Timedelta(FREQ))  # 96
DAYS = 7
TOTAL_STEPS = STEPS_PER_DAY * DAYS                           # 672 steps

# Output
OUT_DIR = "."  # Current directory
OUT_CSV = os.path.join(OUT_DIR, f"dhaka_timeseries_{SIM_START_DATE}_7d_15min.csv.gz")

# Safety knobs for prototyping
SAMPLE_EDGES = 12000   # e.g., 5000 to limit edges while testing; None = all edges
WRITE_HEADER = True

# Speed model (free-flow) & dynamics
MIN_SPEED_KPH = 2.0             # realistic Dhaka jam
MIN_SPEED_FRACTION = 0.20
NOISE_STD = 0.12                # higher randomness
WEEKEND_DAYS = {4, 5}           # Fri=4, Sat=5 (Mon=0)

# Dhaka-specific free-flow speeds (km/h)
BD_HWY_SPEED_KPH = {
    "motorway": 70, "trunk": 55, "primary": 40, "secondary": 30, "tertiary": 25,
    "unclassified": 20, "residential": 15, "living_street": 10, "service": 12,
    "motorway_link": 45, "trunk_link": 40, "primary_link": 35,
    "secondary_link": 25, "tertiary_link": 20
}

# Congestion intensity by road type (Dhaka-adjusted)
ROADTYPE_CONG_MULT = {
    "motorway": 1.05, "trunk": 1.10,
    "primary": 1.20, "secondary": 1.35, "tertiary": 1.65,
    "residential": 1.75, "living_street": 2.0, "service": 1.50
}

# ===========================================
# 2) Download, simplify, and strongly-connect the graph
# ===========================================
print("Downloading OSM graph for:", PLACE)
G = ox.graph_from_place(PLACE, network_type=NETWORK_TYPE, simplify=True, retain_all=True)

# Keep the largest strongly connected component (directed)
G = ox.utils_graph.get_largest_component(G, strongly=True)

# Fix: Remove problematic maxspeed attributes that cause TypeError
for u, v, k, data in G.edges(keys=True, data=True):
    if "maxspeed" in data:
        del data["maxspeed"]

# Add/Impute speeds and compute free-flow travel times
G = ox.add_edge_speeds(G, hwy_speeds=BD_HWY_SPEED_KPH)  # adds 'speed_kph'
# Edge 'length' is already meters.

# Normalize 'highway' -> 'road_type', copy to 'free_speed_kph'
for u, v, k, data in G.edges(keys=True, data=True):
    hwy = data.get("highway")
    if isinstance(hwy, list): hwy = hwy[0]
    data["road_type"] = hwy if isinstance(hwy, str) else str(hwy)
    sp = data.get("speed_kph", None)
    try:
        data["free_speed_kph"] = float(sp) if sp is not None else np.nan
    except Exception:
        data["free_speed_kph"] = np.nan

# Fill remaining NaNs from fallback map
for u, v, k, data in G.edges(keys=True, data=True):
    if not np.isfinite(data["free_speed_kph"]):
        data["free_speed_kph"] = float(BD_HWY_SPEED_KPH.get(data["road_type"], 30.0))

# ===========================================
# 3) Static edge table
# ===========================================
def edge_rows(G):
    for u, v, k, d in G.edges(keys=True, data=True):
        yield {
            "u": u, "v": v, "key": k,
            "length_m": float(d.get("length", np.nan)),
            "free_speed_kmh": float(d.get("free_speed_kph", np.nan)),
            "road_type": d.get("road_type", "unknown"),
        }

edges_df = pd.DataFrame(edge_rows(G))
if SAMPLE_EDGES is not None and SAMPLE_EDGES < len(edges_df):
    edges_df = edges_df.sample(SAMPLE_EDGES, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"Edges prepared: {len(edges_df):,}")

# ===========================================
# 4) Dynamic traffic functions
# ===========================================
def base_time_of_day_factor(hour: int) -> float:
    """Approximate congestion factor based on Dhaka time-of-day."""
    if 0 <= hour < 6:   return 0.15
    if 6 <= hour < 8:   return 0.70   # morning rush
    if 8 <= hour < 10:  return 0.90
    if 10 <= hour < 12: return 0.60
    if 12 <= hour < 14: return 0.55   # Friday spike handled separately
    if 14 <= hour < 16: return 0.50
    if 16 <= hour < 19: return 0.95   # evening peak
    if 19 <= hour < 21: return 0.70
    if 21 <= hour < 23: return 0.40
    return 0.15


def traffic_factor(ts: pd.Timestamp, road_type: str) -> float:
    """Traffic factor based on time-of-day and road type."""
    hour = ts.hour
    dow = ts.dayofweek
    factor = base_time_of_day_factor(hour)

    # Weekend adjustment
    if dow in WEEKEND_DAYS:
        factor *= 0.60
        # Friday mid-day Jumu'ah spike
        if dow == 4 and 12 <= hour < 14:
            factor *= 1.25

    # Apply road-type congestion multiplier
    factor *= ROADTYPE_CONG_MULT.get(road_type, 1.0)

    # Noise scaling by minor road types for stochastic realism
    noise_scale = NOISE_STD
    if road_type in ["residential", "living_street"]:  # use road_type, not rt
        noise_scale *= 1.5
    factor = np.clip(factor + np.random.normal(0.0, noise_scale), 0.05, 1.00)

    return float(factor)  # make sure to return the factor



def factor_to_speed(free_kph: float, factor: float) -> float:
    """Convert traffic factor to current speed (km/h)."""
    max_reduction = 1.0 - MIN_SPEED_FRACTION
    multiplier = 1.0 - max_reduction * factor
    return float(max(free_kph * multiplier, MIN_SPEED_KPH))

# ===========================================
# 5) Seven-day time index (Asia/Dhaka)
# ===========================================
time_index = pd.date_range(
    start=pd.Timestamp(SIM_START_DATE).tz_localize(TZ),
    periods=TOTAL_STEPS,
    freq=FREQ
)
print(f"Timestamps: {len(time_index):,} (from {time_index[0]} to {time_index[-1]})")

# ===========================================
# 6) Stream to ONE CSV (GZIP)
# ===========================================
if os.path.exists(OUT_CSV):
    os.remove(OUT_CSV)

header = ["timestamp","u","v","key","road_type",
          "length_m","free_speed_kmh","traffic_factor",
          "current_speed_kmh","travel_time_seconds"]

with gzip.open(OUT_CSV, "wt", newline="") as gzfile:
    writer = csv.writer(gzfile)
    if WRITE_HEADER:
        writer.writerow(header)

    for ts in tqdm(time_index, desc="Writing CSV over time (7 days)"):
        for row in edges_df.itertuples(index=False):
            length_m = float(row.length_m)
            free_kph  = float(row.free_speed_kmh)
            rt = row.road_type

            f = traffic_factor(ts, rt)
            cur_kph = factor_to_speed(free_kph=free_kph, factor=f)
            cur_mps = cur_kph * (1000.0 / 3600.0)
            t_sec = length_m / cur_mps if cur_mps > 0 else math.inf

            writer.writerow([
                ts.isoformat(), row.u, row.v, row.key, rt,
                round(length_m, 3), round(free_kph, 2), round(f, 4),
                round(cur_kph, 2), round(t_sec, 2)
            ])

print("Done. Wrote:", OUT_CSV)

# ===========================================
# 7) Quick peek
# ===========================================
pd.read_csv(OUT_CSV, nrows=10, compression="gzip")

d:\OneDrive\Desktop\AUST\Thesis_docs\.venv\Lib\site-packages\osmnx\_overpass.py:254: UserWarning: This area is 11 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_16008\299972457.py:51: FutureWarning: The `get_largest_component` function is deprecated and will be removed in the v2.0.0 release. Replace it with `truncate.largest_component` instead. See the OSMnx v2 migration guide: https://github.com/gboeing/osmnx/issues/1123
  G = ox.utils_graph.get_largest_component(G, strongly=True)


Edges prepared: 12,000
Timestamps: 672 (from 2025-02-15 00:00:00+06:00 to 2025-02-21 23:45:00+06:00)


Writing CSV over time (7 days): 100%|██████████| 672/672 [06:32<00:00,  1.71it/s]

Done. Wrote: .\dhaka_timeseries_2025-02-15_7d_15min.csv.gz


,timestamp,u,v,key,road_type,length_m,free_speed_kmh,traffic_factor,current_speed_kmh,travel_time_seconds
0,2025-02-15T00:00:00+06:00,7858277362,7858277421,0,unclassified,98.208,20.0,0.0500,19.20,18.41
1,2025-02-15T00:00:00+06:00,5017920339,5017920343,0,residential,322.910,15.0,0.3329,11.01,105.63
2,2025-02-15T00:00:00+06:00,9902261157,9902261208,1,residential,224.443,15.0,0.2272,12.27,65.83
3,2025-02-15T00:00:00+06:00,5010756319,5010757630,0,living_street,91.206,10.0,0.0916,9.27,35.43
4,2025-02-15T00:00:00+06:00,5591097883,9169375006,0,tertiary,324.916,25.0,0.2937,19.13,61.16
5,2025-02-15T00:00:00+06:00,7876547114,7876547170,0,residential,203.300,15.0,0.0500,14.40,50.83
6,2025-02-15T00:00:00+06:00,5606845061,5606844778,0,residential,59.015,15.0,0.1661,13.01,16.33
7,2025-02-15T00:00:00+06:00,3320720067,4447620198,0,residential,250.773,15.0,0.3136,11.24,80.34
8,2025-02-15T00:00:00+06:00,7858285676,7858285618,0,unclassified,440.909,20.0,0.0500,19.20,82.67
9,2025-02-15T00:00:00+06:00,5660572454,5639230270,0,residential,178.166,15.0,0.0500,14.40,44.54


In [5]:
df = pd.read_csv(OUT_CSV, compression="gzip")
df.tail(10)

,timestamp,u,v,key,road_type,length_m,free_speed_kmh,traffic_factor,current_speed_kmh,travel_time_seconds
8063990,2025-02-21T23:45:00+06:00,7773978569,7876529047,0,residential,319.438,15.0,0.6767,6.88,167.17
8063991,2025-02-21T23:45:00+06:00,9075629180,9075629178,0,trunk,264.077,55.0,0.0852,51.25,18.55
8063992,2025-02-21T23:45:00+06:00,4772513281,4772508777,0,residential,93.295,15.0,0.0995,13.81,24.33
8063993,2025-02-21T23:45:00+06:00,6163073812,6163074050,0,unclassified,126.712,20.0,0.1670,17.33,26.33
8063994,2025-02-21T23:45:00+06:00,1794726907,10027894542,0,unclassified,283.287,20.0,0.0500,19.20,53.12
8063995,2025-02-21T23:45:00+06:00,2313085859,4948358164,0,residential,21.673,15.0,0.4969,9.04,8.63
8063996,2025-02-21T23:45:00+06:00,10262961895,10262961867,0,residential,151.621,15.0,0.2128,12.45,43.86
8063997,2025-02-21T23:45:00+06:00,4413033982,9901595803,0,residential,4.981,15.0,0.1260,13.49,1.33
8063998,2025-02-21T23:45:00+06:00,5606189279,5657966468,0,unclassified,880.847,20.0,0.2728,15.63,202.82
8063999,2025-02-21T23:45:00+06:00,5647308873,5647178922,0,residential,40.266,15.0,0.0500,14.40,10.07
